# Test Suite — HJB Replication Project

Runs the full `pytest tests/` suite and displays a visual pass/fail summary.

**Usage:** Run all cells top-to-bottom, or use *Run All* (`⇧↩` on last cell).

> Requires `pytest` — install once with `pip install pytest`.

In [ ]:
import subprocess, sys, os, re
from pathlib import Path
from IPython.display import display, HTML

# Make sure we're in the project root
ROOT = Path(globals().get('__file__', '')).parent
if not ROOT.is_absolute():
    ROOT = Path.cwd()
os.chdir(ROOT)
print(f'Working dir: {ROOT}')


In [ ]:
def render_results(stdout: str, returncode: int):
    """Parse pytest -v output into a coloured HTML table."""
    PASS_COL  = '#d4edda'  # green
    FAIL_COL  = '#f8d7da'  # red
    SKIP_COL  = '#fff3cd'  # yellow
    HEAD_COL  = '#e2e3e5'

    rows = []
    pattern = re.compile(
        r'(?P<path>tests/[\w/]+\.py)::(?P<cls>[\w]+)::(?P<fn>[\w]+)\s+(?P<status>PASSED|FAILED|ERROR|SKIPPED)'
    )
    for line in stdout.splitlines():
        m = pattern.search(line)
        if m:
            st = m.group('status')
            col = PASS_COL if st=='PASSED' else (FAIL_COL if st in ('FAILED','ERROR') else SKIP_COL)
            icon = '✅' if st=='PASSED' else ('❌' if st in ('FAILED','ERROR') else '⏭️')
            rows.append(f'<tr style="background:{col}">'
                        f'<td>{m.group("path")}</td>'
                        f'<td>{m.group("cls")}</td>'
                        f'<td>{m.group("fn")}</td>'
                        f'<td>{icon} {st}</td></tr>')

    # Summary line
    summary_re = re.search(r'(\d+ passed.*)', stdout)
    summary = summary_re.group(1) if summary_re else ('FAILED' if returncode else 'OK')
    summary_col = '#d4edda' if returncode == 0 else '#f8d7da'

    table = (
        '<table style="width:100%;border-collapse:collapse;font-family:monospace;font-size:0.85em">'
        f'<tr style="background:{HEAD_COL};font-weight:bold">'
        '<th style="text-align:left;padding:4px">File</th>'
        '<th style="text-align:left;padding:4px">Class</th>'
        '<th style="text-align:left;padding:4px">Test</th>'
        '<th style="text-align:left;padding:4px">Result</th></tr>'
        + ''.join(rows)
        + '</table>'
        + f'<p style="background:{summary_col};padding:6px;font-weight:bold">{summary}</p>'
    )
    return table


In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short', '--color=no'],
    capture_output=True, text=True, cwd=str(ROOT)
)
stdout = result.stdout + result.stderr
display(HTML(render_results(stdout, result.returncode)))
if result.returncode != 0:
    print('\n── Full output ──')
    print(stdout)


In [ ]:
# Optional: show raw pytest output for debugging
# Uncomment the next line to print full output
# print(stdout)


In [ ]:
# Run individual test files and show per-module pass rates
modules = [
    ('fd_core',       'tests/test_fd_core.py'),
    ('data_loader',   'tests/test_data_loader.py'),
    ('backtest_core', 'tests/test_backtest_core.py'),
    ('nn_core',       'tests/test_nn_core.py'),
]

rows = []
for label, path in modules:
    r = subprocess.run(
        [sys.executable, '-m', 'pytest', path, '-v', '--tb=no', '--color=no', '-q'],
        capture_output=True, text=True, cwd=str(ROOT)
    )
    out = r.stdout
    n_pass = len(re.findall(r'PASSED', out))
    n_fail = len(re.findall(r'FAILED|ERROR', out))
    n_skip = len(re.findall(r'SKIPPED', out))
    total  = n_pass + n_fail + n_skip
    pct    = 100 * n_pass / total if total else 0
    bar_w  = int(pct * 1.5)
    bar    = f'<div style="background:#28a745;width:{bar_w}px;height:12px;display:inline-block"></div>'
    icon   = '✅' if n_fail==0 and total>0 else ('⚠️' if n_skip==total else '❌')
    rows.append(
        f'<tr><td style="padding:4px">{icon} {label}</td>'
        f'<td style="padding:4px">{n_pass}/{total}</td>'
        f'<td style="padding:4px">{n_skip} skipped</td>'
        f'<td style="padding:4px">{bar} {pct:.0f}%</td></tr>'
    )

html = (
    '<h3>Per-module summary</h3>'
    '<table style="font-family:monospace;border-collapse:collapse">'
    '<tr style="background:#e2e3e5;font-weight:bold">'
    '<th style="padding:4px">Module</th><th style="padding:4px">Passed</th>'
    '<th style="padding:4px">Skipped</th><th style="padding:4px">Coverage</th></tr>'
    + ''.join(rows) + '</table>'
)
display(HTML(html))
